# SDGi Corpus: Report-Grouped Dataset

The official splits are combined before report grouping. Each row remains a passage with its original SDG labels and metadata.

### Why report grouping?

The original baseline audit found no exact or normalized duplicate texts, but 587 of 588 validation passages and 1,459 of 1,470 test passages shared a source report with training. Related wording and context can make evaluation optimistic.

Keeping each inferred report in one split reduces this overlap. These results are reported separately from the original passage-split baseline.


## 1. Import libraries and set paths


In [ ]:
import os
from itertools import combinations
import re
import unicodedata

import numpy as np
import pandas as pd
from IPython.display import display

OUTPUT_DIR = "/Users/aman/Desktop/Thesis/sdgi_report_grouped/data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SEED = 42

## 2. Load and clean passages


In [ ]:
train_df = pd.read_parquet("/Users/aman/Desktop/Thesis/raw datasets/sdgi-corpus/data/train-00000-of-00001.parquet", columns=["text", "labels", "metadata"])
test_df = pd.read_parquet("/Users/aman/Desktop/Thesis/raw datasets/sdgi-corpus/data/test-00000-of-00001.parquet", columns=["text", "labels", "metadata"])

print("Official training passages:", len(train_df))
print("Official test passages:", len(test_df))

train_df["official_split"] = "train"
test_df["official_split"] = "test"
all_df = pd.concat([train_df, test_df], ignore_index=True)

all_df["text"] = all_df["text"].fillna("").apply(lambda text: re.sub(r"\s+", " ", text).strip())
print("Empty passages removed:", all_df["text"].eq("").sum())
all_df = all_df[all_df["text"].ne("")].copy()

print("Total passages:", len(all_df))
display(all_df.head())

## 3. Identify source reports

The report_id comes from metadata.file_id after removing the SDG folder and final numeric passage suffix. For example, sdg-02/example-report-2.pdf and sdg-04/example-report-4.pdf both become example-report.pdf. Passages from different SDG folders can therefore share a report_id.

The rest of the filename, including year and language, is retained. Language versions remain separate report IDs. Ignoring language identified eight candidate report families across splits, so grouping does not guarantee separation of translations or differently named versions.


In [ ]:
def get_report_id(metadata):
    file_id = metadata["file_id"]
    filename = file_id.rsplit("/", 1)[-1]
    if not re.search(r"-\d+\.pdf$", filename):
        raise ValueError(f"Unexpected filename: {file_id}")
    return re.sub(r"-\d+\.pdf$", ".pdf", filename)

all_df["report_id"] = all_df["metadata"].apply(get_report_id)
report_sizes = all_df.groupby("report_id").size()

print("Source reports:", len(report_sizes))
display(all_df[["report_id", "labels"]].head())

## 4. Create report-grouped splits

Reports are shuffled with seed 42, sorted from largest to smallest, and assigned to the split with the largest remaining passage deficit. Equal deficits are resolved in train, validation, test order. Assignment uses report sizes only.

The passage targets are 80% training, 10% validation and 10% test. Whole-report grouping can prevent exact percentages.


In [ ]:
report_order = np.random.default_rng(SEED).permutation(report_sizes.index)
report_order = sorted(report_order, key=lambda report: report_sizes[report], reverse=True)

split_names = ["train", "validation", "test"]
target_counts = np.array([0.80, 0.10, 0.10]) * len(all_df)
actual_counts = np.zeros(3, dtype=int)
report_split = {}

for report in report_order:
    split_index = int(np.argmax(target_counts - actual_counts))
    report_split[report] = split_names[split_index]
    actual_counts[split_index] += int(report_sizes[report])

all_df["split"] = all_df["report_id"].map(report_split)
train_df = all_df[all_df["split"].eq("train")].copy()
validation_df = all_df[all_df["split"].eq("validation")].copy()
test_df = all_df[all_df["split"].eq("test")].copy()

splits = {"train": train_df, "validation": validation_df, "test": test_df}
summary = pd.DataFrame([
    {"split": name, "passages": len(frame),
     "percentage": round(100 * len(frame) / len(all_df), 2),
     "reports": frame["report_id"].nunique()}
    for name, frame in splits.items()
])
display(summary)

## 5. Check overlap and SDG coverage


In [ ]:
def normalized_text(text):
    text = unicodedata.normalize("NFKC", text).casefold()
    return re.sub(r"[^\w]+", " ", text).strip()

assert sum(len(frame) for frame in splits.values()) == len(all_df)
assert all(len(frame) > 0 for frame in splits.values())
assert all_df.groupby("report_id")["split"].nunique().eq(1).all()

for left_name, right_name in combinations(split_names, 2):
    left, right = splits[left_name], splits[right_name]
    assert set(left["report_id"]).isdisjoint(right["report_id"]), "Report overlap"
    assert set(left["text"]).isdisjoint(right["text"]), "Exact text overlap"
    assert set(left["text"].apply(normalized_text)).isdisjoint(
        right["text"].apply(normalized_text)
    ), "Normalized text overlap"

print("Checks passed: all passages assigned; no report or duplicate text overlap.")

coverage = pd.DataFrame(index=pd.Index(range(1, 18), name="SDG"))
for name, frame in splits.items():
    coverage[name] = [frame["labels"].apply(lambda labels: goal in labels).sum()
                      for goal in coverage.index]

In [ ]:
coverage

## 6. Save the dataset

Exports retain text, labels, metadata, official_split, report_id and split. Original embeddings are omitted.


In [ ]:
for name, frame in splits.items():
    frame.to_parquet(f"{OUTPUT_DIR}/{name}.parquet", index=False)

summary.to_csv(f"{OUTPUT_DIR}/split_summary.csv", index=False)
coverage.to_csv(f"{OUTPUT_DIR}/sdg_coverage.csv")

print("Dataset saved to:", OUTPUT_DIR)